<a href="https://colab.research.google.com/github/IsmayilGasim/AI-Lesson-notes/blob/main/Inception_model_with_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch.nn as nn
import torch.nn.functional as F
import torch

In [49]:
# class InceptionBlock(nn.Module):
#   def __init__(self,
#                in_channel,
#                c1,
#                c3_reduce, c3,
#                c5_reduce, c5,
#                out_pool):
#     super().__init__()

#     # branch 1
#     self.c1 = nn.Conv2d(in_channel, c1, 1,padding="same")

#     # branch 2
#     self.c2 = nn.Sequential(
#               nn.Conv2d(in_channel, c3_reduce, 1, padding="same"),
#               nn.ReLU(),
#               nn.Conv2d(c3_reduce, c3, 3, padding=1) )

#     # branch 3
#     self.c3 = nn.Sequential(
#               nn.Conv2d(in_channel, c5_reduce, 1, padding="same"),
#               nn.ReLU(),
#               nn.Conv2d(c5_reduce, c5, 5, padding=2))

#     # branch 4
#     self.c4 = nn.Sequential(
#               nn.MaxPool2d(3,1,1),
#               nn.Conv2d(in_channel, out_pool, 1, padding="same"))

#   def forward(self, X):
#       out1 = F.relu(self.c1(X))
#       out2 = F.relu(self.c2(X))
#       out3 = F.relu(self.c3(X))
#       out4 = F.relu(self.c4(X))

#       return torch.concat([out1,out2,out3,out4],dim=1)

class InceptionBlock(nn.Module):
  def __init__(self, in_channel, c1,
               c3_reduce, c3,
               c5_reduce, c5,
               out_pool):
    super().__init__()

    # branch 1
    self.c1 = nn.Conv2d(in_channel, c1, 1, padding='same')

    # branch 2
    self.c2 = nn.Sequential(
              nn.Conv2d(in_channel, c3_reduce, 1, padding='same'),
              nn.ReLU(),
              nn.Conv2d(c3_reduce, c3, 3, padding=1))
    # branch 3
    self.c3 = nn.Sequential(
              nn.Conv2d(in_channel, c5_reduce, 1, padding='same'),
              nn.ReLU(),
              nn.Conv2d(c5_reduce, c5, 5, padding=2))

    # branch 4
    self.c4 = nn.Sequential(
        nn.MaxPool2d(3, 1, 1),
        nn.Conv2d(in_channel, out_pool, 1, padding='same')
    )

  def forward(self, X):
    out1 = F.relu(self.c1(X))
    out2 = F.relu(self.c2(X))
    out3 = F.relu(self.c3(X))
    out4 = F.relu(self.c4(X))

    return torch.concat([out1, out2, out3, out4], dim=1)

In [50]:
# class InceptionNet(nn.Module):
#   def __init__(self,in_channels, n_classes: int=10):
#     super().__init__()

#     # conv 1
#     self.conv1 = nn.Conv2d(in_channels, 64, 7, stride=2, padding=3)
#     self.pool1 = nn.MaxPool2d(3,2)

#     # conv 2
#     self.conv2 = nn.Conv2d(64, 192, 1, stride=1,padding='valid')
#     self.pool2 = nn.MaxPool2d(3,2)

#     # inception 1
#     self.inception_3a = InceptionBlock(192, 64, 96, 128, 16, 32, 32)
#     self.inception_3b = InceptionBlock(256, 128, 128, 192, 32, 96, 64)

#     # pool 3
#     self.pool3 = nn.MaxPool2d(3, 2,)

#     # inception 2
#     self.inception_4a = InceptionBlock(480, 192, 96, 208, 16, 48, 64)
#     self.inception_4b = InceptionBlock(512, 116, 120, 224, 24, 64, 64)
#     self.inception_4c = InceptionBlock(512, 128, 128, 256, 24, 64, 64)
#     self.inception_4d = InceptionBlock(512, 112, 144, 288, 32, 64, 64)
#     self.inception_4e = InceptionBlock(528, 256, 160, 320, 32, 128, 128)

#     # pool 4
#     self.pool4 = nn.MaxPool2d(3, 2,)

#     # inception 3
#     self.inception_5a = InceptionBlock(832, 256, 160, 320, 32, 128, 128)
#     self.inception_5b = InceptionBlock(832, 384, 192, 384, 48, 128, 128)

#     # average pool 1
#     self.avg_pool1 = nn.AdaptiveAvgPool2d(1024)

#     # dropout
#     self.dropout = nn.Dropout(0.4)

#     # outputs
#     self.main_output = nn.Linear(1024, n_classes)
#     # self.aux_output1 = nn.Sequential(
#     #     nn.AvgPool2d(5,3),
#     #     nn.Conv2d(aux1, 256, 1, 'same'),
#     #     nn.ReLU(),
#     #     nn.Avg
#     # )

#   def forward(self, X):
#       x = F.relu(self.conv1(X))
#       x = self.pool1(x)

#       x = F.relu(self.conv2(x))
#       x = self.pool2(x)

#       x = self.inception_3a(x)
#       x = self.inception_3b(x)

#       x = self.pool3(x)

#       x = self.inception_4a(x)
#       x = self.inception_4b(x)
#       x = self.inception_4c(x)
#       x = self.inception_4d(x)
#       x = self.inception_4e(x)

#       x = self.pool4(x)

#       x = self.inception_5a(x)
#       x = self.inception_5b(x)


#       x = self.avg_pool1(x)
#       x = self.dropout(x)
#       x = self.main_output(x)

#       return x






In [80]:
class InceptionNet(nn.Module):
  def __init__(self, in_channels, n_classes: int=10):   #aux1, aux2,
    super().__init__()

    # conv 1
    self.conv1 = nn.Conv2d(in_channels, 64, 7, stride=2, padding=3 )
    self.pool1 = nn.MaxPool2d(3, 2)

    # conv 2
    self.conv2 = nn.Conv2d(64, 192, 1, stride=1, padding='valid')
    self.pool2 = nn.MaxPool2d(3, 2, )

    # inception 1
    self.inception_3a = InceptionBlock(192, 64, 96, 128, 16, 32, 32)
    self.inception_3b = InceptionBlock(256, 128, 128, 192, 32, 96, 64) ### 224

    # pool 3
    self.pool3 = nn.MaxPool2d(3, 2, )

    # inception 2
    self.inception_4a = InceptionBlock(480, 192, 96, 208, 16, 48, 64)
    self.inception_4b = InceptionBlock(512, 160, 112, 224, 24, 64, 64)
    self.inception_4c = InceptionBlock(512, 128, 128, 256, 24, 64, 64)
    self.inception_4d = InceptionBlock(512, 112, 144, 288, 32, 64, 64)
    self.inception_4e = InceptionBlock(528, 256, 160, 320, 32, 128, 128)

    # pool 4
    self.pool4 = nn.MaxPool2d(3, 2, )

    # inception 3
    self.inception_5a = InceptionBlock(832, 256, 160, 320, 32, 128, 128)
    self.inception_5b = InceptionBlock(832, 384, 192, 384, 48, 128, 128)

    # average pool 1
    self.avg_pool1 = nn.AdaptiveAvgPool2d(1024)
    self.dropout = nn.Dropout(0.4)

    # outputs
    self.main_output = nn.Linear(1024, n_classes)

    self.aux_output1 = nn.Sequential(
        nn.AdaptiveAvgPool2d(480),
        nn.Conv2d(480, 256, 1),
        nn.ReLU(),
        nn.Linear(480,10),
        nn.ReLU(),
        nn.Linear(10, n_classes)
    )

    self.aux_output2 = nn.Sequential(
        nn.AdaptiveAvgPool2d(832),
        nn.Conv2d(832, 256, 1),
        nn.ReLU(),
        nn.Linear(832,10),
        nn.ReLU(),
        nn.Linear(10, n_classes)
    )

  def forward(self, X):
    x = F.relu(self.conv1(X))
    x = self.pool1(x)

    x = F.relu(self.conv2(x))
    x = self.pool2(x)

    # inception 1
    x = self.inception_3a(x)
    x = self.inception_3b(x)

    x = self.pool3(x)

    aux_output1 = self.aux_output1(x)

    # inception 2
    x = self.inception_4a(x)
    x = self.inception_4b(x)
    x = self.inception_4c(x)
    x = self.inception_4d(x)
    x = self.inception_4e(x)

    x = self.pool4(x)

    aux_output2 = self.aux_output2(x)


    # inception 3

    x = self.inception_5a(x)
    x = self.inception_5b(x)

    # avg pool
    x = self.avg_pool1(x)
    x = self.dropout(x)
    x = self.main_output(x)

    return x, aux_output1, aux_output2

In [81]:
model  = InceptionNet(3)

In [24]:
#!/bin/bash
!curl -L -o fracture-multi-region-x-ray-data.zip https://www.kaggle.com/api/v1/datasets/download/bmadushanirodrigo/fracture-multi-region-x-ray-data

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  481M  100  481M    0     0   252M      0  0:00:01  0:00:01 --:--:--  297M


In [25]:
!unzip /content/fracture-multi-region-x-ray-data.zip

Streaming output truncated to the last 5000 lines.
  inflating: Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/fractured/80-rotated3-rotated3 - Copy.jpg  
  inflating: Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/fractured/80-rotated3-rotated3-rotated1 (1).jpg  
  inflating: Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/fractured/80-rotated3-rotated3-rotated1 - Copy (1).jpg  
  inflating: Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/fractured/80-rotated3-rotated3-rotated1 - Copy.jpg  
  inflating: Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/fractured/80-rotated3-rotated3-rotated1.jpg  
  inflating: Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/fractured/80-rotated3-rotated3-rotated2 - Copy (1).jpg  
  inflating: Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/fract

In [30]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [31]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()])

In [68]:
dataset = datasets.ImageFolder('/content/Bone_Fracture_Binary_Classification/train', transform=transform)

In [69]:
train_loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)

In [70]:
model = InceptionNet(3,2)

In [71]:
dataset.classes

['fractured', 'not fractured']

In [72]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

In [73]:
import matplotlib as plt

In [74]:
def train_inception(model, criterion, optimizer, data_loader, n_epochs, plot=False):
  for epoch in range(n_epochs):
    total_loss = 0.
    losses = []
    for images, labels in data_loader:
      images, labels = images.to(device), labels.to(device)
      pred_labels = model(images)
      loss = criterion(pred_labels, labels)

      total_loss += loss
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

    mean_loss = total_loss/len(data_loader)
    losses.append(mean_loss)


    print(f'Epoch: {epoch+1}/{n_epochs}, Loss: {mean_loss}')

    if plot:
      plt.title('Loss Curve')
      plt.plot(range(n_epochs), losses)
      plt.show




In [75]:
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
xentropy = nn.BCELoss()

In [76]:
train_inception(model, xentropy, optimizer, train_loader, 20, True)

OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 GiB. GPU 0 has a total capacity of 14.56 GiB of which 10.62 GiB is free. Including non-PyTorch memory, this process has 3.94 GiB memory in use. Of the allocated memory 3.80 GiB is allocated by PyTorch, and 21.13 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
x = torch.randn(1,3,224, 224)
y=model(x)

In [ ]:
# resnet 34 alqoritmin yazmaq